# AgentReach AI: Needle 3 on Google Colab

This notebook tests the AgentReach read-only prototype using Cactus Compute's `cactus-needle` package. Run the cells **from top to bottom**.

## What this test includes
- Needle 3 tool calling in the current Colab runtime.
- Read-only lookup of a **public** GitHub repository (`karpathy/nanochat`).
- A report of integrations that are not connected yet.

No paid inference API key or GitHub token is needed for these public read-only tests. The first agent initialization can download a small runtime/model asset, so internet access is required. Colab usage and availability limits still apply.

**Safety boundary:** this notebook does not access private repositories, run arbitrary shell commands, modify GitHub, read local files, or connect Gmail/Drive/Calendar. Those require separate implementations and permissions.

In [ ]:
%pip install -q cactus-needle requests

## 2. Check the runtime and import Needle

This cell also disables optional anonymous usage telemetry for this session. It does not change how tool calls work.

In [ ]:
import os
os.environ["NEEDLE_TELEMETRY"] = "0"
os.environ["DO_NOT_TRACK"] = "1"

import sys
import platform
import json
import requests
import needle
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("Runtime:", platform.platform())
try:
    print("cactus-needle:", version("cactus-needle"))
except PackageNotFoundError:
    print("cactus-needle distribution version not found")
print("Needle import: OK")

## 3. Define the allow-listed read-only tools

The decorators use explicit triggers so the matching request is more likely to route to the intended tool. Each tool has a narrow scope.

In [ ]:
@needle.tool(triggers=[r"(?i)\bgithub\b"])
def get_public_github_repo(owner: str, repo: str) -> dict:
    """Get metadata for a public GitHub repository. Provide the owner and repository name."""
    allowed = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_."
    if (not owner or not repo
        or any(ch not in allowed for ch in owner + repo)
        or owner in {".", ".."} or repo in {".", ".."}):
        return {"ok": False, "error": "Invalid GitHub owner or repository name."}

    try:
        response = requests.get(
            f"https://api.github.com/repos/{owner}/{repo}",
            headers={
                "Accept": "application/vnd.github+json",
                "User-Agent": "AgentReach-Colab-Prototype",
            },
            timeout=15,
        )
        if response.status_code == 404:
            return {"ok": False, "error": "Public repository not found or not accessible."}
        response.raise_for_status()
        data = response.json()
        return {
            "ok": True,
            "name": data.get("full_name"),
            "description": data.get("description"),
            "url": data.get("html_url"),
            "language": data.get("language"),
            "stars": data.get("stargazers_count"),
            "default_branch": data.get("default_branch"),
            "license": (data.get("license") or {}).get("spdx_id"),
        }
    except requests.RequestException as exc:
        return {"ok": False, "error": f"GitHub request failed: {type(exc).__name__}"}


@needle.tool(triggers=[r"(?i)\bintegrations?\b", r"(?i)\bwhich apps are connected\b"])
def list_supported_integrations() -> dict:
    """List currently available and not-yet-connected app integrations."""
    return {
        "available": [
            "Needle 3 tool-calling runtime",
            "Public GitHub repository metadata (read-only, no account token)",
        ],
        "not_connected": [
            "Private GitHub access and write actions",
            "Gmail", "Google Drive", "Google Calendar",
            "Browser automation", "Local file access",
        ],
        "note": "Each additional app must be set up explicitly with least-privilege permissions.",
    }

print("Two read-only tools defined.")

## 4. Verify public GitHub API connectivity without the AI model

This separates a network/API problem from a Needle inference problem.

In [ ]:
direct_check = get_public_github_repo("karpathy", "nanochat")
print(json.dumps(direct_check, indent=2, ensure_ascii=False))
assert direct_check.get("ok") is True, "Public GitHub lookup failed; inspect the error above."

## 5. Initialize Needle 3

The first initialization may fetch the inference engine. If this cell errors, keep the full error text so we can diagnose runtime/platform compatibility.

In [ ]:
agent = needle.Needle(tools=[get_public_github_repo, list_supported_integrations])
print("Needle agent initialized. Tools available:", [get_public_github_repo.__name__, list_supported_integrations.__name__])

## 6. Run an AI-directed public repository lookup

The agent should select the public GitHub lookup tool, execute it, and return a result object. This is the first end-to-end tool-call test.

In [ ]:
lookup_result = agent.run(
    "Use the GitHub tool to look up the public repository karpathy/nanochat and show its metadata."
)
print(json.dumps(lookup_result, indent=2, ensure_ascii=False, default=str))
if isinstance(lookup_result, dict):
    print("\nReturned keys:", list(lookup_result.keys()))
    print("\nExecuted tool results:", json.dumps(lookup_result.get("results", []), indent=2, ensure_ascii=False, default=str))

## 7. Test the integration-status tool

In [ ]:
integration_result = agent.run("Which integrations are currently available, and which apps are not connected?")
print(json.dumps(integration_result, indent=2, ensure_ascii=False, default=str))

## What success looks like

1. The package imports and prints a version.
2. The direct GitHub API cell returns `"ok": true` for `karpathy/nanochat`.
3. `needle.Needle(...)` initializes without an exception.
4. The agent result includes the public repository metadata in its executed `results`.

If a cell fails, stop at that cell and save the complete traceback. Do not paste passwords, access tokens, or private account data into chat. These tests do **not** yet connect Gmail, Drive, Calendar, private GitHub, or app write operations.